## Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Darknet Kurulumu ve Kütüphaneler

In [ ]:
%cd /content/
!git clone https://github.com/AlexeyAB/darknet

In [ ]:
!pip install tensorflow
!pip install opencv-python
!pip install rsa
!pip install cryptography
!pip install bcrypt
!pip install scrypt

In [ ]:
%cd darknet
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
!sed -i 's/LIBSO=0/LIBSO=1/' Makefile
!make

In [4]:
import tensorflow
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import *
import os
import cv2
import keras
import time
from cryptography.fernet import Fernet
import bcrypt
import secrets
import scrypt
import base64



from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode, b64encode
import PIL
import io
import html
import time
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1V3vsIaxAlGWvK4Aar9bAiK5U0QFttKwq' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1V3vsIaxAlGWvK4Aar9bAiK5U0QFttKwq" -O yolov4-csp.weights && rm -rf /tmp/cookies.txt

In [6]:
!pwd
%cd /content/darknet
from darknet import *

network, class_names, class_colors = load_network("/content/darknet/cfg/yolov4-csp.cfg", "/content/darknet/cfg/coco.data", "/content/darknet/yolov4-csp.weights")

/content/darknet
/content/darknet


In [7]:
width = network_width(network)
height = network_height(network)

In [8]:
def darknet_helper(img, width, height):
  darknet_image = make_image(width, height , 3)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  img_resized = cv2.resize(img_rgb, (width,height), interpolation = cv2.INTER_LINEAR)

  img_height, img_width, _ = img.shape
  width_ratio = img_width/width   #resim ve ağ yukseklik-genislik oranı
  height_ratio = img_height / height
  copy_image_from_bytes(darknet_image, img_resized.tobytes())   
  detections = detect_image(network, class_names, darknet_image)
  free_image(darknet_image)
  return detections, width_ratio, height_ratio

In [9]:
def js_to_image(js_reply):

  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8) 
  img= cv2.imdecode(jpg_as_np, flags=-1) 
  return img

In [10]:
def bbox_to_bytes(bbox_array):

  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA') 
  iobuf = io.BytesIO()

  bbox_PIL.save(iobuf, format='png')
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()),'utf-8')))
  return bbox_bytes

In [11]:
def video_stream():
  js = Javascript('''  
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;
    
    var pendingResolve = null;
    var shutdown = false;
    
    function removeDom() {    //butona basildiginda silinmesi icin kullanilan fonksiyon
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }
    
    function onAnimationFrame() { //video nun cizdirildigi fonksiyon
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }
    
    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);
      
      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);
           
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);
      
      const instruction = document.createElement('div');
      instruction.innerHTML = 
          '<span style="color: red; font-weight: bold;">' +
          'Detect Area</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };
      
   
      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; 
      captureCanvas.height = 480; 
      window.requestAnimationFrame(onAnimationFrame);
      
      return stream;
    }
    async function stream_frame(label, imgData, flag) {   
      if (shutdown) {
        removeDom();
        if(flag){
          shutdown = true;
        }else{
          shutdown = false;
        }
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();
      
      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }
            
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }
      
      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;
      
      return {'create': preShow - preCreate, 
              'show': preCapture - preShow, 
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)


  
def video_frame(label, bbox, flag):   
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox, flag))
  if flag == True:
    return True
  return data

In [12]:
video_stream()
label_html = 'Capturing..'
bbox = ''
count = 0

<IPython.core.display.Javascript object>

In [21]:
key_path = "/content/key.csv"
enc_message_file = "/content/enc_message_file.csv"
password_path = "/content/secret.txt"


def encode(pswrd):

  message = input("\U0001F4C3 Mesaj : ")  

  salt = secrets.token_bytes(32)
  key = scrypt.hash(pswrd, salt, N=2048, r=8, p=1, buflen=32)
  key = base64.urlsafe_b64encode(key)


  # kullanılan fernet anahtarı key.csv icine kaydedilir. 
  # Bu anahtar, kullanicinin girecegi --> verinin <-- sifrelenmesi ve sifrenin cozulmesi icin kullanilir

  # fernet key is save into key.csv.
  # This key is used for encrypt and decrypt the --> data <-- entered by the user
  with open( key_path , 'wb') as filekey:
      filekey.write(key)
  
  print("\n\n\U0001F512 \U0001F60E Fernet key saved.. ")
  fernet = Fernet(key)
  enc_message = fernet.encrypt(message.encode())
  print("\U0001F4C3 Message will write into enc_message_file.csv : ", enc_message)

  with open(enc_message_file, "wb") as enc_messages:
    enc_messages.write(enc_message)
  print("\U0001F60E Message saved...\n\n")
  

def decode():
  
  with open(key_path, "rb") as key_file :
    key = key_file.readline()

  with open(enc_message_file, "r") as enc_messages:
    enc_message = enc_messages.readline()

  fernet = Fernet(key)
  dec_message = str(fernet.decrypt(enc_message).decode())
  print("\n\n\U0001F4C3 Message : ", dec_message,"\n\n")

In [ ]:
import datetime

video_stream()
label_html = 'Capturing..'
bbox = ''
count = 0
images_dir = "/content/drive/MyDrive/Login"
label = ""
flag = False

while True:

    js_reply = video_frame(label_html, bbox, flag)
    if not js_reply or flag == True:
        break

    frame = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480,640,4], dtype=np.uint8)

    # tespit değerleri alınır
    # take detection values
    detections, width_ratio, height_ratio = darknet_helper(frame, width, height)

    for label, confidence, bbox in detections:
        left, top, right, bottom = bbox2points(bbox)

        # tespit kutusuna ait değerler elde edilir.
        left, top, right, bottom = int(left * width_ratio), int(top * height_ratio), int(right * width_ratio), int(bottom * height_ratio)

        # tespit kutularını çizdirmek için aşağıdaki kod satırını açabilirsiniz
        # you can open this command line for draw detection boxes
        #bbox_array = cv2.rectangle(bbox_array, (left, top), (right, bottom), class_colors[label], 2)
        bbox_array = cv2.putText(bbox_array, "{} [{:.2f}]".format(label, float(confidence)),
                          (left, top - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                          class_colors[label], 2)
            

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes


    # sisteme sadece insan sınıfı tespiti yapılırsa girilebilir. 
    # only people class use this app
    if label == "person":   
          print("\U0001F467\U0001F466 Label: ", label)
          # tespit edilen kişiye ait görüntü alınır ve Login adlı klasöre kaydedilir
          # The image of the detected person is taken and saved in the folder named Login.
          time = datetime.datetime.today()
          cv2.imwrite(os.path.join(images_dir, "detect_person_" + str(time) +".jpg"), frame)
         

          # dogru password "merhaba dunya"
          # correct password is "merhaba dunya"
          check = open(password_path,"r+")


          # sifreli password ile karsilastirma yapilmasi icin kullanicidan password alinir 
          # Password is taken from the user for compare with the encrypted password
          passw = input("\n\n\U0001F92B Password: ")
          pswrd = bytes(passw, 'utf-8')
          # sifreli password, secret.txt icinden alinir
          # encrypted password is taken  from secret.txt file
          txt_datas=check.read().split('\n')
          check_pass = bytes(txt_datas[0],'utf-8')
          # kullanici password u ile karsilastirma yapilir
          # Comparing with the user password
          control_pswrd = bcrypt.checkpw(pswrd, check_pass)


          # password doğru ise ;
          # If password is correct ; 
          if control_pswrd:
            operation = int(input("\n\n\U0001F4DD Please select the action : \n1)encode,\n2)decode\n"))

            # yeni bir girdi alarak şifreler ve enc_message_file.csv içine kaydeder. Eski veri silinir.
            # takes a new entry, encrypts it and saves it in enc_message_file.csv. Old data is deleted.
            if operation == 1 : 
              encode(pswrd)

            # enc_message_file.csv içindeki şifreli metin çevirilerek gösterilir. Burada kayıt yapılmaz. Veri sadece çıktıda görülebilir.
            # The ciphertext in enc_message_file.csv is translated and displayed. No registration here. The data is only visible in the terminal output.
            else :
              decode()
            flag = True

          else:
            print("\n\U0001F480 Wrong password !! \n\n")
            flag = True
    